# SAFIRA | Seção 4.2.1: Exploração de dados

**Inteli, Turma T28, Grupo 01** | Parceiro: **Azul Linhas Aéreas Brasileiras**

Implementação da exploração de dados documentada na seção 4.2.1 de `documents/documentacao.md`. O objetivo é caracterizar a estrutura, a qualidade e a representatividade das bases fornecidas pela Azul antes da etapa de preparação, conforme a fase de *Data Understanding* da metodologia CRISP-DM (CHAPMAN et al., 2000).

A unidade de análise adotada é **o voo avaliado**, e não o Cliente. O alvo é a detração no NPS principal, binarizada em Detrator contra não-Detrator.

Cada figura nasce da execução da célula imediatamente acima dela. Nenhuma imagem é incorporada como arquivo pronto.

### Uma fonte de verdade

Este notebook **não reimplementa** a lógica de limpeza, estatística ou plotagem. Ele importa as funções de `src/` e as orquestra, intercalando a leitura de cada resultado.

A razão é de consistência, e não de estilo. Enquanto o mesmo cálculo existia nos dois lugares, as duas cópias divergiam sem que nada avisasse: a tabela de categóricas do notebook chegou a listar 15 variáveis contra as 17 da documentação, porque era montada antes de as derivadas existirem. Com uma implementação só, uma mudança em `src/` se propaga ao notebook e à documentação ao mesmo tempo.

| Módulo | Responsabilidade |
|---|---|
| `src/clean.py` | Carga, integração validada, filtros e variáveis derivadas |
| `src/stats.py` | Classificação de colunas, tabelas descritivas, V de Cramér e diagnóstico dos pesos |
| `src/graficos.py` | As sete figuras, com o tema visual institucional |

**Ordem de execução contra ordem de leitura.** A documentação apresenta as variáveis derivadas no item (f), depois da estatística descritiva do item (c). Em execução a ordem tem que ser a inversa, porque `FAIXA_ATRASO`, `N_TRECHOS` e `AEROPORTO_ORIGEM` aparecem nas tabelas descritivas. O mapa entre as duas ordens:

| Seção do notebook | Item da documentação |
|---|---|
| 2. Carga e integração | (a) Estrutura e integração das bases |
| 3. Filtros e duplicidades | (b) Filtros aplicados |
| 4. Variáveis derivadas | (f) Variáveis derivadas |
| 5. Unidade de análise | (b) Registro sobre a unidade de análise |
| 6. Classificação e estatística descritiva | (c) Classificação e estatística descritiva |
| 7. Qualidade dos dados | (d) Inconsistências identificadas |
| 8. Representatividade e viés | (e) Representatividade e vieses da amostra |
| 9. Relações entre variáveis | (g) Análise das relações entre variáveis |
| 10. Síntese | (i) Síntese e implicações |

> ### Aviso sobre os dados
>
> As bases da Azul **não são versionadas neste repositório**, por restrição do Termo de Abertura de Projeto de Inovação. O diretório `data/` está no `.gitignore` e nenhum arquivo `.csv`, `.pkl` ou `.parquet` é publicado.
>
> Para reproduzir, aponte `CAMINHO_DADOS` na célula de configuração para a pasta local que contém os cinco arquivos, ou monte o Google Drive no Colab. As saídas de célula que exibiriam linhas individuais de Cliente foram mantidas apenas em forma agregada.

---

## 1. Configuração do ambiente

### 1.1. Importação da implementação canônica

A raiz do projeto é localizada subindo a árvore de diretórios até encontrar `src/`. Isso faz o notebook funcionar tanto executado da raiz quanto de dentro de `notebooks/`, que é o diretório de trabalho que o JupyterLab e o `nbconvert --execute` usam por padrão. No Colab, monte o Drive e defina `RAIZ` manualmente.

In [ ]:
import sys
from pathlib import Path


def raiz_projeto(inicio=None):
    'Sobe a arvore de diretorios ate encontrar a pasta src/.'
    atual = Path(inicio or Path.cwd()).resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / 'src').is_dir():
            return candidato
    raise FileNotFoundError(
        f'src/ nao encontrado a partir de {atual}. Defina RAIZ manualmente.')


RAIZ = raiz_projeto()
# No Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# RAIZ = Path('/content/drive/MyDrive/SAFIRA/g01')

CAMINHO_SRC = RAIZ / 'src'
sys.path.insert(0, str(CAMINHO_SRC))

import inspect
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from clean import (ORD_ATRASO, carregar_bases, carregar_populacao, deduplicar,
                   derivar, integrar, pesos_pos_estratificacao)
from stats import (CATEGORICAS, CAT_ASSOCIACAO, NUMERICAS, classificar_colunas,
                   comparar_bruto_ponderado, cramers_v, diagnostico_pesos,
                   sensibilidade_antecedencia, tabela_antecedencia,
                   tabela_categoricas, tabela_cramer, tabela_numericas)
from graficos import (LAB_ATRASO, aplicar_tema, virgula,
                      g1_atraso_dose_resposta, g2_serie_temporal,
                      g3_detracao_por_tier, g5_correlacao, g7_limiar_atraso,
                      g8_antecedencia_cancelamento, g9_sazonalidade)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
aplicar_tema()

print('seaborn', sns.__version__, '| matplotlib', plt.matplotlib.__version__,
      '| pandas', pd.__version__)
print(f'implementacao canonica carregada de {CAMINHO_SRC}')

### 1.2. Divisão entre as bibliotecas de visualização

A stack combina as duas bibliotecas, e não por redundância:

- **`seaborn`** fixa o tema visual e a paleta institucional em todas as figuras via `set_theme`, e segue respondendo pela gramática estatística nos gráficos 1, 7, 8 e 9 (`barplot`, `lineplot`, e `relplot` nos pequenos múltiplos do gráfico 9).
- **`matplotlib`** entra com mais peso nos gráficos 2, 3 e 5, cujo desenho editorial (cabeçalho com antetítulo, painéis numerados, rótulos ao fim de cada linha, matriz triangular com legenda própria) exige controle posicional mais fino do que a gramática estatística do `seaborn` abstrai. Em todos os casos ele também responde pelo eixo secundário (`twinx`) nos gráficos 1 e 8, anotações posicionais, formatação dos ticks com vírgula decimal e composição de subplots com proporções assimétricas no gráfico 7.

A escolha reflete a arquitetura das bibliotecas: o `seaborn` (WASKOM, 2021) é construído sobre o `matplotlib` (HUNTER, 2007) e o uso conjunto é o padrão recomendado.

A célula abaixo exibe o código de uma das figuras, para que a construção fique visível aqui e não apenas no módulo.

In [ ]:
print(inspect.getsource(g1_atraso_dose_resposta))

### 1.3. Caminho das bases e destino das figuras

In [ ]:
CAMINHO_DADOS = RAIZ / 'dados'
# No Colab: CAMINHO_DADOS = Path('/content/drive/MyDrive/SAFIRA/dados')

# Os PNGs recebem os mesmos nomes que documentacao.md referencia em assets/.
DIR_FIGURAS = RAIZ / 'figuras'


def mostrar(fig, nome):
    'Renderiza a figura como saida da celula e grava o PNG correspondente.'
    if DIR_FIGURAS:
        os.makedirs(DIR_FIGURAS, exist_ok=True)
        fig.savefig(os.path.join(DIR_FIGURAS, f'{nome}.png'), dpi=200, bbox_inches='tight')
    plt.show()


assert Path(CAMINHO_DADOS).is_dir(), (
    f'{CAMINHO_DADOS} nao encontrado. Aponte CAMINHO_DADOS para a pasta com os cinco arquivos.')

print(f'raiz do projeto : {RAIZ}')
print(f'bases           : {CAMINHO_DADOS}')
print(f'figuras         : {DIR_FIGURAS}')
sorted(p.name for p in Path(CAMINHO_DADOS).glob('*.csv'))

---

## 2. Carga e integração das bases

*Corresponde ao item (a) da documentação.*

O material recebido é composto por cinco arquivos, com duas naturezas distintas.

As três primeiras fontes são **transacionais** e compartilham a chave `RESPONDENT_ID` em relação **1:1**. `NPS_01` a `NPS_04` trazem a pesquisa de satisfação, `INFORMACAO_VIAGEM` os dados operacionais do voo e `PERFIL_CLIENTE_01` e `_02` o perfil de relacionamento.

A quarta fonte, `DISTRIBUICAO_PAX_NORMALIZADO`, é **agregada** e não transacional. Ela informa a composição real do universo de passageiros por mês, faixa de atraso e canal de compra, e por isso serve de referência populacional para o diagnóstico de viés da seção 8.

In [ ]:
nps, perfil, viagem = carregar_bases(CAMINHO_DADOS)
dist = carregar_populacao(CAMINHO_DADOS)

for nome, tabela in [('NPS', nps), ('PERFIL', perfil), ('VIAGEM', viagem),
                     ('DIST PAX', dist)]:
    print(f'{nome:<9}: {tabela.shape[0]:>7} linhas x {tabela.shape[1]:>3} colunas')

---

## 3. Filtros, duplicidades e integração validada

*Corresponde ao item (b) da documentação.*

A verificação produziu um resultado atipicamente limpo, o que por si só informa sobre a maturidade do processo de extração da Azul.

| Filtro | Justificativa |
|---|---|
| F1. Linhas integralmente duplicadas | Duplicação de ingestão. Aplicado às três tabelas |
| F2. Chave duplicada com conteúdo divergente | Só prossegue se a divergência estiver em coluna previamente aprovada. Qualquer outra interrompe a execução |
| F3a. Cobertura da chave | Confere a correspondência antes da junção, porque um `inner join` descartaria respostas em silêncio |
| F3. Junção interna 1:1 | `validate="one_to_one"` do pandas, que interrompe se a cardinalidade declarada não se sustentar |
| F4. Colunas constantes | `VOO_INTERNACIONAL` assume `Domestic` em 100% dos registros, sem poder discriminativo |

Duas decisões merecem destaque, porque a alternativa cômoda seria silenciosa.

A primeira: **não se usa `drop_duplicates(subset='RESPONDENT_ID')` às cegas**. Manter a primeira ocorrência sem olhar o conteúdo descartaria dado válido sem aviso e tornaria o total de registros uma contagem sem significado. Chave repetida com conteúdo divergente só é tolerada na coluna que a documentação registra como caso aprovado; qualquer outra levanta exceção.

A segunda: **a relação 1:1 é verificada, não presumida**, via `validate="one_to_one"`.

In [ ]:
log = []
df = integrar(nps, perfil, viagem, log)

Uma trava que nunca dispara não prova nada. A célula abaixo injeta uma duplicata divergente artificial e verifica que a integração recusa prosseguir. É teste, não transformação: nada é alterado na base real.

In [ ]:
teste = nps.head(2).copy()
teste.iloc[1, teste.columns.get_loc('RESPONDENT_ID')] = teste.iloc[0]['RESPONDENT_ID']
teste.iloc[1, teste.columns.get_loc('VOO_TIPO')] = 'VALOR_DIVERGENTE'

try:
    deduplicar(teste, 'teste_sintetico')
    print('FALHOU: a divergencia passou sem bloqueio')
except ValueError as erro:
    print('Trava funcionou. Mensagem recebida:')
    print(f'  {str(erro)[:200]}...')

---

## 4. Variáveis derivadas

*Corresponde ao item (f) da documentação.*

Esta etapa vem antes da estatística descritiva por necessidade de execução: três das variáveis derivadas aparecem nas tabelas da seção 6.

| Variável | Origem | Justificativa |
|---|---|---|
| `N_TRECHOS` | Contagem de separadores em `BASE_AIRPORTLEG` | Complexidade do itinerário |
| `AEROPORTO_ORIGEM` e `AEROPORTO_DESTINO` | Decomposição de `BASE_AIRPORTLEG` | Reduz a cardinalidade de milhares de itinerários para poucas centenas de aeroportos |
| `FAIXA_ATRASO` | Discretização de `ESTATISTICA_ATRASOSAIDA` | Compatibiliza com a taxonomia da Azul e com a base populacional |
| `MES_ANO` e `TRIMESTRE` | Extração de `DATA_STD` | Capturam o efeito sazonal e o efeito de período |
| `DETRATOR` e `CLASSE_NPS` | Recodificação de `NPS_PRINCIPAL` | Alvo binarizado e rótulo das três classes |

`PESO_POP` é construída apenas na seção 8, por depender da base populacional.

In [ ]:
df = derivar(df, log)

print('\nCardinalidade antes e depois da decomposicao:')
print(f'  BASE_AIRPORTLEG : {df["BASE_AIRPORTLEG"].nunique()} itinerarios')
print(f'  AEROPORTO_ORIGEM: {df["AEROPORTO_ORIGEM"].nunique()} aeroportos')
print('\nDistribuicao de FAIXA_ATRASO:')
print(df['FAIXA_ATRASO'].value_counts(normalize=True).mul(100).round(2).to_string())

### 4.1. Relação de `N_TRECHOS` com o alvo

A hipótese é que a complexidade do itinerário seja, isoladamente, um fator de risco.

In [ ]:
print('Taxa de detratores por numero de trechos:')
print(df.groupby('N_TRECHOS')['DETRATOR']
        .agg(n='size', taxa=lambda s: s.mean() * 100).round(2).to_string())
print(f'\nQuatro ou mais trechos: n={int((df["N_TRECHOS"] >= 4).sum())}, '
      f'taxa={virgula(df[df["N_TRECHOS"] >= 4]["DETRATOR"].mean() * 100, 2, "%")}')

A relação é **monotônica** e a variável está disponível no momento da predição, sem risco de vazamento. O patamar de quatro trechos reúne os itinerários de quatro a seis, porque acima de quatro o volume é residual.

### 4.2. Derivação descartada: `ASSENTOS` como tamanho de grupo

Este registro é de rigor metodológico, não erro a esconder. O campo `ASSENTOS` foi inicialmente interpretado como indicador do tamanho do grupo viajante. A verificação cruzada refutou a hipótese, e o teste está reproduzido abaixo.

In [ ]:
cruzada = pd.crosstab(df['N_ASSENTOS_AUDIT'], df['N_TRECHOS'])
print('Tabulacao cruzada N_ASSENTOS_AUDIT x N_TRECHOS:')
print(cruzada.to_string())

par = df[['N_ASSENTOS_AUDIT', 'N_TRECHOS']].dropna()
print(f'\nCorrelacao de Spearman: {virgula(par.corr(method="spearman").iloc[0, 1], 3)}')
print(f'Casos na diagonal (contagens iguais): '
      f'{virgula((par["N_ASSENTOS_AUDIT"] == par["N_TRECHOS"]).mean() * 100, 1, "%")}')

A correspondência é quase perfeita. O registro `20A/17A/28A`, associado ao itinerário `FOR/UDI/CNF/POA`, corresponde a **três assentos do mesmo passageiro em três trechos consecutivos**, e não a três passageiros.

A variável foi mantida apenas como campo de auditoria e **excluída do conjunto de preditores por redundância** com `N_TRECHOS`. Nenhum campo da base permite identificar viagens em grupo, limitação registrada como pedido de dado adicional ao parceiro.

---

## 5. Unidade de análise e dependência intracliente

*Corresponde ao registro sobre unidade de análise do item (b).*

Embora não haja duplicidade de chave, o volume de respostas é maior que o de Clientes: parte dos respondentes participou da pesquisa mais de uma vez. A decisão de manter ou colapsar esses registros define a unidade de análise de todo o projeto.

In [ ]:
recorrentes = (df['N_RESP_CLIENTE'] > 1).mean() * 100
print(f'Respostas             : {len(df)}')
print(f'Clientes distintos    : {df["ID_GOLDENRECORD"].nunique()}')
print(f'Respostas recorrentes : {virgula(recorrentes, 1, "%")}')
print(f'Maximo de respostas por Cliente: {int(df["N_RESP_CLIENTE"].max())}')

varia = df.groupby('ID_GOLDENRECORD')['DETRATOR'].nunique()
print(f'\nClientes que detrataram em um voo e nao em outro: {int((varia > 1).sum())}')

print('\nTaxa de detracao por numero de respostas do Cliente:')
faixa_resp = pd.cut(df['N_RESP_CLIENTE'], [0, 1, 3, 100],
                    labels=['1 resposta', '2 a 3', '4 ou mais'])
print(df.groupby(faixa_resp, observed=True)['DETRATOR'].mean().mul(100).round(2).to_string())

Os registros repetidos **não foram removidos**, por duas razões. Primeiro, a unidade de decisão da área de Customer Experience é o voo: o mesmo Cliente pode detratar em um voo com atraso e promover no seguinte, e colapsar por Cliente eliminaria exatamente a variação intraindividual que o modelo precisa aprender. Segundo, a recorrência não distorce a variável resposta, como mostra a tabela acima.

A célula seguinte quantifica a dependência que resta, e que impõe uma restrição à modelagem.

In [ ]:
dois = df[df['N_RESP_CLIENTE'] == 2]
observado = (dois.groupby('ID_GOLDENRECORD')['DETRATOR'].sum() == 2).mean() * 100
esperado = dois['DETRATOR'].mean() ** 2 * 100

print(f'Detrataram nas duas respostas, observado: {virgula(observado, 1, "%")}')
print(f'Esperado sob independencia (p^2)        : {virgula(esperado, 1, "%")}')
print(f'Razao observado / esperado              : {virgula(observado / esperado, 1, "x")}')

print('\nComposicao por tier, respondentes unicos vs. recorrentes:')
print(pd.crosstab(df['N_RESP_CLIENTE'] > 1, df['TIER_VIAGEM'], normalize='index')
      .mul(100).round(1).rename(index={False: 'Unico', True: 'Recorrente'}).to_string())

nulos_gold = int(df['ID_GOLDENRECORD'].isna().sum())
print(f'\nRegistros sem ID_GOLDENRECORD: {nulos_gold} '
      f'({virgula(nulos_gold / len(df) * 100, 2, "%")})')

> **Restrição derivada para a fase de modelagem.** A partição entre treino e teste deverá ser **agrupada por `ID_GOLDENRECORD`** (`GroupKFold` ou `GroupShuffleSplit`). Uma partição aleatória simples permitiria que o mesmo Cliente figurasse em ambos os conjuntos, levando o modelo a memorizar padrões individuais e superestimando artificialmente as métricas de desempenho.
>
> Ressalva: os registros sem `ID_GOLDENRECORD` não podem ser agrupados por Cliente e são **excluídos da validação** por `dividir_treino_teste_temporal_por_cliente` (`scripts/preprocessamento_nps.py`), que os deixa fora do treino e do teste, registra a exclusão em log e informa o total nos metadados da partição. Tratá-los como grupos unitários suporia que são Clientes distintos, o que a base não permite afirmar, já que o nulo ocorre na pesquisa e no perfil ao mesmo tempo.

---

## 6. Classificação das colunas e estatística descritiva

*Corresponde ao item (c) da documentação.*

### 6.1. Classificação programática

A separação entre numéricas e categóricas é feita **por código**, e não por lista escrita à mão. O critério não é apenas o `dtype`: campos numéricos com no máximo duas categorias, como flags, são tratados como categóricos, porque a média deles não tem interpretação de escala.

In [ ]:
num_auto, cat_auto, temp_auto = classificar_colunas(df)
print(f'Numericas  : {len(num_auto)}')
print(f'Categoricas: {len(cat_auto)}')
print(f'Temporais  : {len(temp_auto)}')
print()
print(inspect.getsource(classificar_colunas))

Da classificação automática são separados os conjuntos que efetivamente entram na análise descritiva. As listas vêm de `src/stats.py` e são as mesmas que geram as tabelas da documentação.

Ficam de fora os identificadores e os campos `NPS_*` de etapa da jornada, que são coletados no mesmo instrumento que origina a variável resposta e, portanto, configurariam vazamento se usados como preditores, conforme estabelecido na seção 4.1.3.

In [ ]:
vazamento = [c for c in df.columns if c.startswith('NPS_') and c != 'NPS_PRINCIPAL']

print(f'Numericas analisadas   : {len(NUMERICAS)}')
print(f'Categoricas analisadas : {len(CATEGORICAS)}')
print(f'Associacao com o alvo  : {len(CAT_ASSOCIACAO)}')
print(f'Excluidas por vazamento: {len(vazamento)} colunas {vazamento[:6]} ...')

### 6.2. Variáveis numéricas

Além dos momentos usuais, a tabela traz o P95, a nulidade e a assimetria. A assimetria é o que decide o tratamento na fase de preparação: valores altos indicam que média e desvio padrão descrevem mal a distribuição.

In [ ]:
tabela_numericas(df)

Todas as variáveis numéricas apresentam **forte assimetria positiva**, com mediana consistentemente inferior à média. Nos campos de atraso a mediana é zero, o que reflete uma operação majoritariamente pontual.

A consequência metodológica é dupla: transformações logarítmicas ou discretização em faixas serão preferíveis ao uso das variáveis em escala bruta, e métricas baseadas em média descrevem mal esses campos.

### 6.3. Variáveis categóricas

Para as categóricas, o que interessa é a cardinalidade, a concentração na moda e a nulidade. Cardinalidade muito alta inviabiliza codificação categórica convencional.

In [ ]:
tabela_categoricas(df)

`BASE_AIRPORTLEG`, `ASSENTOS`, `EQUIPAMENTO_TIPO` e `VOO_NUMERO` apresentam cardinalidade muito elevada. Os três primeiros são campos compostos, cujo valor informacional foi extraído por decomposição na seção 4. `VOO_NUMERO`, por identificar operações específicas, será descartado do conjunto de preditores.

### 6.4. Distribuição do alvo

`NPS_PRINCIPAL` assume três valores, correspondentes às classes definidas por Reichheld (2003) na formulação original do Net Promoter Score.

In [ ]:
print(df['CLASSE_NPS'].value_counts(normalize=True).mul(100).round(2).to_string())
print(f'\nNPS bruto: {virgula(df["NPS_PRINCIPAL"].mean(), 1)}')
print(f'Desbalanceamento Detrator vs. nao-Detrator: '
      f'1 para {virgula((1 - df["DETRATOR"].mean()) / df["DETRATOR"].mean(), 1)}')

### 6.5. Força de associação das categóricas com o alvo

O coeficiente de correlação não se aplica a variáveis nominais. A medida adequada é o **V de Cramér** (CRAMÉR, 1946), derivado do qui-quadrado de contingência e normalizado para o intervalo de 0 a 1.

A tabela é única e já inclui as derivadas, porque elas existem nesta altura da execução.

In [ ]:
print(inspect.getsource(cramers_v))
tabela_cramer(df)

Nenhuma variável categórica isolada apresenta associação forte com a detração. O maior valor corresponde à faixa de atraso, variável derivada. O resultado reforça o caráter **multivariado** do fenômeno e justifica a escolha de algoritmos capazes de capturar interações.

Vale destacar desde já o contraste que a seção 8 desenvolve: o canal de compra é o segundo maior eixo de viés amostral, mas praticamente não discrimina o alvo. Ele explica **quem responde**, não **quem detrata**.

---

## 7. Qualidade dos dados: inconsistências identificadas

*Corresponde ao item (d) da documentação.*

Quatro achados, um por célula. Nenhum deles é erro de ingestão: todos exigem decisão metodológica explícita na fase de preparação.

### 7.1. Nulidade estrutural em `TIPO_ENTRETENIMENTO`

A hipótese é que a ausência não decorre de falha de coleta, mas da própria natureza do voo. A tabulação cruzada com `VOO_TIPO` testa isso.

In [ ]:
nulos_ent = df['TIPO_ENTRETENIMENTO'].isna()
print(f'Nulos em TIPO_ENTRETENIMENTO: {int(nulos_ent.sum())} '
      f'({virgula(nulos_ent.mean() * 100, 2, "%")})')
print('\nTabulacao cruzada com VOO_TIPO:')
print(pd.crosstab(df['VOO_TIPO'], nulos_ent).rename(
    columns={False: 'Preenchido', True: 'Nulo'}).to_string())

print('\nANTECEDENCIA_CANCELAMENTO condicionada a CANCELAMENTO_VOO:')
print(pd.crosstab(df['CANCELAMENTO_VOO'],
                  df['ANTECEDENCIA_CANCELAMENTO'].isna()).rename(
    columns={False: 'Preenchido', True: 'Nulo'}).to_string())

A correspondência é exata: os registros nulos são precisamente os voos de conexão. Como uma conexão envolve mais de uma aeronave, não existe um único sistema de entretenimento associado ao trecho, e a imputação seria conceitualmente incorreta. O tratamento adequado é a criação de uma categoria explícita, `Não aplicável (conexão)`.

O mesmo raciocínio vale para `ANTECEDENCIA_CANCELAMENTO`: o campo está preenchido em 100% dos voos cancelados e nulo em 100% dos não cancelados, sendo **condicionado** a `CANCELAMENTO_VOO` e não faltante.

### 7.2. Divergência entre os campos de atraso

`ESTATISTICA_ATRASOSAIDA` e `ATRASO_CHEGADA` deveriam medir o mesmo evento operacional em dois momentos. A correlação entre eles testa se de fato medem.

In [ ]:
rho = df[['ESTATISTICA_ATRASOSAIDA', 'ATRASO_CHEGADA']].corr(method='spearman').iloc[0, 1]
print(f'Correlacao de Spearman entre os campos de atraso: {virgula(rho, 3)}')

print(f'Partida pontual e chegada com mais de 60 min de atraso: '
      f'{int(((df["ESTATISTICA_ATRASOSAIDA"] < 15) & (df["ATRASO_CHEGADA"] > 60)).sum())}')
print(f'Partida atrasada e chegada registrada como pontual    : '
      f'{int(((df["ESTATISTICA_ATRASOSAIDA"] > 60) & (df["ATRASO_CHEGADA"] == 0)).sum())}')

print(f'\nATRASO_CHEGADA igual a zero: '
      f'{virgula((df["ATRASO_CHEGADA"] == 0).mean() * 100, 1, "%")}')
print(f'Maximo de ATRASO_CHEGADA   : {int(df["ATRASO_CHEGADA"].max())} min')
print('\nMedia de ATRASO_CHEGADA por status do voo:')
print(df.groupby('CANCELAMENTO_VOO')['ATRASO_CHEGADA'].mean().round(1).to_string())

A hipótese de trabalho é que o campo **agrega semânticas distintas**: chegada antecipada codificada como zero, e tempo até a reacomodação nos casos de cancelamento. A observação é consistente com o apontamento da própria equipe da Azul quanto à necessidade de revisão dos campos usados em situações de atraso.

**A validação da regra de cálculo de `ATRASO_CHEGADA` foi encaminhada ao ponto focal da Azul como pendência.**

### 7.3. Inconsistência entre frequência declarada e observada

`SUB_FIL_FREQUENCIAAZUL` é autodeclarada pelo respondente. `QTDE_VIAGENS_36M` vem do registro operacional. Confrontar as duas mede o erro do instrumento.

In [ ]:
primeira = df[df['SUB_FIL_FREQUENCIAAZUL'].astype(str)
              .str.contains('primeira', case=False, na=False)]
tiers_altos = ['Diamante', 'Safira', 'Topazio']

print(f'Declararam "Esta foi a primeira vez": {len(primeira)}')
if len(primeira):
    print(f'  com mais de uma viagem em 36 meses: '
          f'{virgula((primeira["QTDE_VIAGENS_36M"] > 1).mean() * 100, 1, "%")}')
    print(f'  pertencentes a Diamante, Safira ou Topazio: '
          f'{virgula(primeira["TIER_VIAGEM"].isin(tiers_altos).mean() * 100, 1, "%")}')

print('\nMediana de viagens em 36 meses por frequencia declarada:')
print(df.groupby('SUB_FIL_FREQUENCIAAZUL')['QTDE_VIAGENS_36M'].median().to_string())

A divergência pode refletir ambiguidade na formulação da pergunta, referindo-se à primeira vez naquela rota e não na companhia, ou erro de recordação. Independentemente da causa, o campo apresenta **erro de medida substancial** e, por ser coletado no mesmo instrumento que origina a variável resposta, acumula risco de vazamento.

**Recomenda-se seu descarte em favor de `QTDE_VIAGENS_12M`**, que mensura o mesmo construto a partir de registro operacional.

### 7.4. Outliers em `TEMPO_VOO`

O valor máximo é implausível para operação doméstica. A pergunta é se é erro ou se o campo mede outra coisa.

In [ ]:
print(f'Maximo de TEMPO_VOO: {int(df["TEMPO_VOO"].max())} min '
      f'({virgula(df["TEMPO_VOO"].max() / 60, 1)} horas)')
print('\nDistribuicao por tipo de voo:')
print(df.groupby('VOO_TIPO')['TEMPO_VOO']
        .agg(n='size', mediana='median', p99=lambda s: s.quantile(.99), maximo='max')
        .round(1).to_string())
print(f'\nVoos diretos acima de 600 min: '
      f'{int(((df["VOO_TIPO"] == "Direto") & (df["TEMPO_VOO"] > 600)).sum())}')

A distribuição é aceitável em voos Diretos e apresenta cauda extensa em Conexões. Como `TEMPO_VOO` mede a **viagem completa** e não o tempo em voo, valores elevados em conexões refletem esperas prolongadas: informação legítima e potencialmente preditiva.

O tratamento será por **winsorização no P99 dentro de cada tipo de voo**, e não por exclusão de registros.

---

## 8. Representatividade e viés amostral

*Corresponde ao item (e) da documentação.*

Esta é a contribuição analítica central da exploração. A pesquisa de NPS da Azul é enviada a aproximadamente metade dos Clientes domésticos, com taxa de resposta inferior a 10%. A amostra é, portanto, **autosselecionada**.

A literatura de survey demonstra que baixas taxas de resposta não geram viés por si mesmas, mas o produzem quando a propensão a responder se correlaciona com a variável de interesse (GROVES; PEYTCHEVA, 2008). A base `DISTRIBUICAO_PAX_NORMALIZADO` permite testar exatamente essa condição.

### 8.1. Viés de resposta associado ao atraso

In [ ]:
pop_atraso = dist.groupby('DELAY_DEPARTURE_RANGE')['PERC_PAX'].sum()
pop_atraso = (pop_atraso / pop_atraso.sum() * 100).reindex(ORD_ATRASO)
amo_atraso = df['FAIXA_ATRASO'].value_counts(normalize=True).mul(100).reindex(ORD_ATRASO)

pd.DataFrame({
    '% população PAX': pop_atraso.round(2),
    '% amostra': amo_atraso.round(2),
    'Razão': (amo_atraso / pop_atraso).round(2),
    'Taxa de detratores': df.groupby('FAIXA_ATRASO', observed=True)['DETRATOR']
                            .mean().mul(100).reindex(ORD_ATRASO).round(2),
})

Passageiros que sofreram atraso superior a 120 minutos respondem à pesquisa muito acima do que sua participação na operação justificaria, e detratam a uma taxa várias vezes superior à dos voos pontuais.

Este é o cenário mais crítico de viés amostral: **a propensão a responder está correlacionada com a variável-alvo**.

### 8.2. Viés de canal de compra

In [ ]:
pop_canal = dist.groupby('CANAL_COMPRA')['PERC_PAX'].sum()
pop_canal = pop_canal / pop_canal.sum() * 100
amo_canal = df['CANAL_COMPRA'].value_counts(normalize=True).mul(100)

canal = pd.DataFrame({'% população PAX': pop_canal.round(2), '% amostra': amo_canal.round(2)})
canal['Razão'] = (canal['% amostra'] / canal['% população PAX']).round(2)
canal['Taxa de detratores'] = df.groupby('CANAL_COMPRA')['DETRATOR'].mean().mul(100).round(2)
canal.sort_values('% população PAX', ascending=False)

Clientes que compram via agência estão sub-representados na amostra, enquanto os canais digitais aparecem sobre-representados. O padrão confirma a hipótese levantada pela equipe da Azul de que o contato indireto com a companhia reduz a taxa de resposta.

Note, porém, que a coluna de detração é praticamente plana entre canais. Este é o contraste que a seção 6.5 antecipou: o canal explica **quem responde**, não **quem detrata**.

### 8.3. Pós-estratificação

A pós-estratificação corrige desvios de composição atribuindo a cada observação um peso proporcional à razão entre sua frequência na população e na amostra (VALLIANT, 1993). A estratificação é feita por **mês, faixa de atraso e canal de compra**.

A cobertura integral dessa chave **é condição de parada, não um aviso**. Um estrato da amostra sem contrapartida populacional receberia peso indefinido; preenchê-lo com zero removeria aqueles respondentes do estimador ponderado em silêncio e deslocaria a taxa de detração sem que nada indicasse o problema.

In [ ]:
df = pesos_pos_estratificacao(df, dist, log)

#### Diagnóstico dos pesos

Pesos válidos não bastam: pesos muito desiguais inflam a variância das métricas ponderadas mesmo com cobertura completa. O **tamanho amostral efetivo de Kish** (KISH, 1965), dado por (soma dos pesos)² dividida pela soma dos pesos ao quadrado, responde a quantas observações sem peso esta amostra ponderada equivale.

In [ ]:
diagnostico_pesos(df)

#### Métricas ponderadas com intervalo de confiança

Sem intervalo não é possível afirmar que a diferença entre bruto e ponderado é deslocamento real de composição, e não ruído. O intervalo da média ponderada usa o estimador de Hájek, com variância igual à soma de w²(y − média)² dividida pelo quadrado da soma dos pesos. Ele trata os pesos como fixos e ignora o desenho estratificado, portanto é aproximação, e tende a ser conservador.

In [ ]:
comparar_bruto_ponderado(df)

A amostra bruta **superestima a detração**, e os intervalos de bruto e ponderado não se sobrepõem em nenhuma das duas métricas: a diferença é deslocamento real de composição, não flutuação amostral.

O resultado ganha credibilidade por um teste externo: o NPS pós-estratificado situa-se dentro da faixa de 45 a 50 que a Azul declara como seu patamar corrente, com o intervalo inteiro dentro dessa faixa, enquanto o valor bruto fica abaixo dela.

Conforme decisão da equipe, o viés é **diagnosticado nesta fase e sua incorporação será decidida na etapa de modelagem**.

### 8.4. Efeito de período

A série trimestral testa se a detração varia além do que a operação explica. As duas séries são plotadas juntas justamente para permitir esse contraste.

In [ ]:
mostrar(g2_serie_temporal(df), 'g2_serie_temporal')

O gráfico sozinho não fecha o argumento, porque a incidência de atrasos também sobe no trimestre de pico. A análise condicional abaixo é o que sustenta a leitura de efeito de período.

In [ ]:
por_faixa = (df.pivot_table(index='TRIMESTRE', columns='FAIXA_ATRASO',
                            values='DETRATOR', aggfunc='mean', observed=True) * 100)
print('Taxa de detratores por trimestre, dentro de cada faixa de atraso:')
print(por_faixa.round(1).to_string())

A detração sobe **dentro de todas as faixas**, inclusive entre voos pontuais, onde a pontualidade não pode explicar a variação. O período coincide com o contexto que antecedeu a reestruturação financeira concluída pela companhia, o que sugere componente reputacional externo à operação do voo.

Duas implicações: a validação do modelo deverá adotar **partição temporal**, para não vazar informação de conjuntura entre treino e teste, e a variável temporal deve ser tratada como **covariável de contexto**, não como preditor estável.

### 8.5. Piso irredutível de detração

Isolando o cenário operacionalmente ideal, mede-se quanto da detração **não é operacionalmente evitável**. É o que delimita o teto de desempenho realista do projeto.

In [ ]:
ideal = df[(df['VOO_TIPO'] == 'Direto') & (~df['CANCELAMENTO_VOO'].astype(bool)) &
           (df['ESTATISTICA_ATRASOSAIDA'] < 15) & (df['ATRASO_CHEGADA'] < 15)]

print(f'Registros em condicao operacional ideal: {len(ideal)} '
      f'({virgula(len(ideal) / len(df) * 100, 1, "%")} da base)')
print(f'NPS medio          : {virgula(ideal["NPS_PRINCIPAL"].mean(), 1)}')
print(f'Taxa de detratores : {virgula(ideal["DETRATOR"].mean() * 100, 1, "%")}')
print('\nTaxa de detratores por tier, na condicao ideal:')
print(ideal.groupby('TIER_VIAGEM')['DETRATOR'].mean().mul(100).round(1)
      .sort_values(ascending=False).to_string())

Mesmo na ausência completa de falha operacional, uma parcela relevante dos Clientes avalia a experiência entre 0 e 6. Para a modelagem, isso indica que preditores puramente operacionais terão limite de poder discriminativo.

---

## 9. Análise das relações entre variáveis

*Corresponde ao item (g) da documentação.*

Seis figuras, cada uma respondendo a uma pergunta específica do escopo definido com a Azul. Todas são construídas por funções de `src/graficos.py` e renderizadas como saída de execução. O código de qualquer uma pode ser inspecionado com `inspect.getsource`, como feito na seção 1.2.

> **Nota sobre a natureza das relações.** Esta é uma análise observacional sobre dados de pesquisa autosselecionada. Nenhuma das relações a seguir foi obtida por desenho experimental ou quase-experimental, e portanto **nenhuma delas estabelece causalidade**. A redação adota a forma "está associada a" em vez de "produz" ou "causa", e as recomendações operacionais são apresentadas como **hipóteses a validar**.
>
> Duas limitações sustentam essa cautela: a autosseleção documentada na seção 8, e a ausência na base das variáveis que plausivelmente confundem as relações observadas, com destaque para a **causa do cancelamento e a causa do atraso**.

### Gráfico 1. Atraso na saída: relação com a detração e com o viés de resposta

*Tipo:* barras com eixo secundário. *Variáveis:* `FAIXA_ATRASO` (categórica derivada), taxa de detratores (numérica) e razão de representatividade (numérica).

A figura sobrepõe deliberadamente dois fenômenos que a literatura de pesquisa costuma tratar em separado. As barras (`seaborn.barplot`) mostram o efeito dose-resposta; a linha no eixo secundário (`matplotlib.twinx` mais `seaborn.lineplot`) mostra a sobre-representação da mesma faixa na pesquisa.

In [ ]:
mostrar(g1_atraso_dose_resposta(df, dist), 'g1_atraso_dose_resposta')

**Leitura.** As barras evidenciam um **gradiente monotônico** de magnitude expressiva. O padrão é compatível com uma relação dose-resposta, mas o desenho observacional não permite afirmá-la. A linha revela que essas mesmas faixas são as mais sobre-representadas na pesquisa: a linha pontilhada em 1,00 marca a representatividade perfeita, e todas as faixas com atraso estão acima dela.

A leitura conjunta é o principal insight desta exploração. O atraso é simultaneamente o maior driver de insatisfação e o maior fator de distorção amostral. Qualquer modelo treinado sobre a amostra bruta herdará essa distorção, e qualquer indicador de detração calculado sem ponderação estará inflado.

### Gráfico 7. Limiar de atraso: curva de risco e impacto marginal

*Tipo:* série de linha com painel de variação marginal. *Variáveis:* `ESTATISTICA_ATRASOSAIDA` discretizada em treze faixas e taxa de detratores.

Responde diretamente à pergunta 5 do escopo definido pela Azul: existe um limiar de atraso a partir do qual o risco de detração aumenta significativamente? A composição usa `gridspec` do matplotlib com proporção 2,2 para 1, porque o painel inferior é leitura de apoio e não deve competir visualmente com a curva.

In [ ]:
mostrar(g7_limiar_atraso(df), 'g7_limiar_atraso')

**Leitura.** A resposta é afirmativa e localizável. O painel inferior mostra que **até 15 minutos o custo marginal do atraso é estável**, na ordem de 2 p.p. por faixa. A partir de 20 minutos esse custo dobra e segue acelerando até a faixa de 61 a 90 minutos, quando atinge o máximo. Acima de 180 minutos o incremento desacelera, por efeito de saturação.

A leitura operacional é que **a janela de 20 a 30 minutos é o ponto de maior retorno para a atuação preventiva**. É onde a curva muda de regime e onde a intervenção ainda alcança um contingente grande de Clientes. Recomenda-se que este intervalo seja considerado na definição do *threshold* de acionamento do modelo.

### Gráfico 8. Cancelamento: efeito da antecedência do aviso

*Tipo:* barras com eixo secundário. *Variáveis:* `ANTECEDENCIA_CANCELAMENTO` discretizada, taxa de detratores e NPS médio. Recorte: apenas voos cancelados.

O desenho mantém o evento negativo constante, que é o cancelamento, e varia apenas a comunicação. É o que permite isolar o efeito do aviso do efeito do próprio cancelamento.

In [ ]:
mostrar(g8_antecedencia_cancelamento(df), 'g8_antecedencia_cancelamento')

**Leitura.** É a associação de maior magnitude identificada na exploração. Mantido constante o evento negativo, a antecedência do aviso **está associada a** uma amplitude de dezenas de pontos percentuais na detração e a mais de oitenta pontos de NPS.

O gráfico sozinho não mostra o suporte amostral de cada barra. A tabela abaixo traz o n e o intervalo de confiança de Wilson por faixa, sem os quais a leitura da amplitude entre extremos ignora que as faixas não têm o mesmo suporte.

In [ ]:
tabela_antecedencia(df)

O gradiente é monotônico e os intervalos de faixas adjacentes praticamente não se sobrepõem, exceto entre as três faixas mais longas, onde a curva já estabilizou. A faixa de 1 dia é a de menor suporte amostral e não sustenta leitura isolada.

Note que a estabilização **não** leva a taxa ao patamar geral da base: com mais de 60 dias de aviso, o intervalo permanece inteiramente acima da média geral. O cancelamento avisado com antecedência **continua associado a detração acima da média**, ainda que muito abaixo do aviso de última hora.

#### Análise de sensibilidade

A ressalva central é que a antecedência **não é aleatória**: cancelamentos de mesmo dia decorrem tipicamente de causas operacionais agudas, que carregam transtorno adicional além da falta de aviso.

Para medir quanto da diferença entre os extremos decorre de composição observável, a diferença entre "mesmo dia" e "mais de 60 dias" é recalculada dentro de cada nível de cinco variáveis de controle e ponderada pelo tamanho do estrato. Se encolhesse ao controlar, parte da associação viria de composição.

In [ ]:
sensibilidade_antecedencia(df)

A diferença permanece praticamente inalterada sob todos os controles disponíveis. **Nenhuma das variáveis observáveis explica a associação por composição.** Isso a torna robusta ao que se pode medir, mas não a converte em efeito causal: o fator de confusão mais provável, que é a causa do cancelamento, não existe na base e portanto não pôde ser controlado.

**Hipótese operacional derivada.** A comunicação antecipada do cancelamento é candidata a alavanca de mitigação de alto retorno, consistente com a observação da equipe da Azul de que a comunicação proativa eleva o NPS. Sustentar a recomendação como efeito exigiria controle pela causa e pelo tipo do cancelamento, e idealmente um desenho quase-experimental que comparasse Clientes avisados com antecedências distintas para cancelamentos de causa equivalente. **Fica registrada como hipótese prioritária de validação com o parceiro.**

### Gráfico 3. Taxa de detratores por tier de fidelidade e faixa de atraso

*Tipo:* gráfico de linhas múltiplas. *Variáveis:* `TIER_VIAGEM`, `FAIXA_ATRASO` e taxa de detratores.

Cada linha percorre os tiers de fidelidade para uma faixa de atraso; a pergunta sobre **interação** entre as duas variáveis fica visível no formato das quatro curvas, e não em células isoladas de uma grade. O `seaborn.lineplot` cobre a camada estatística; o `matplotlib` posiciona os rótulos de fim de linha e a anotação do delta por faixa.

In [ ]:
mostrar(g3_detracao_por_tier(df), 'g3_detracao_por_tier')

**Leitura.** O gráfico revela uma **interação entre fidelização e falha operacional** que não seria visível em análises marginais. Em voos pontuais o Cliente Diamante detrata sensivelmente mais que o Cliente sem cadastro. Em voos com mais de 120 minutos de atraso, os dois grupos convergem para o mesmo patamar elevado.

O efeito do tier não é uniforme entre as faixas: ele é maior justamente nas faixas intermediárias e menor nos dois extremos, onde a detração já está baixa (sem atraso) ou já está saturada (acima de 120 min). O padrão geral é consistente com o princípio de que a expectativa de serviço cresce com o nível de relacionamento: **o Cliente mais fidelizado é o menos tolerante à falha**. Para a modelagem, isso indica que `TIER_VIAGEM` e `FAIXA_ATRASO` não devem ser tratadas apenas como efeitos aditivos. Modelos baseados em árvores capturam essa interação naturalmente, enquanto uma regressão logística exigiria termo de interação explícito.

### Gráfico 9. Sazonalidade da detração, controlada por faixa de atraso

*Tipo:* pequenos múltiplos com séries de linha paralelas. *Variáveis:* mês do ano, `FAIXA_ATRASO` e taxa de detratores.

A pergunta metodológica é se a variação mensal decorre apenas da operação ou se há componente sazonal próprio. Os pequenos múltiplos do `seaborn.relplot` resolvem isso por construção: cada painel já é um controle por faixa de atraso.

In [ ]:
mostrar(g9_sazonalidade(df), 'g9_sazonalidade')

**Leitura.** O padrão de dezembro alto e agosto baixo **persiste dentro de todas as quatro faixas**, inclusive entre voos sem atraso algum. A diferença, portanto, não pode ser atribuída à pontualidade.

A hipótese explicativa combina composição de passageiro, com alta concentração de viajantes de lazer e de primeira viagem no período de férias, e congestionamento de infraestrutura, que afeta a experiência sem se traduzir em atraso registrado. A sazonalidade deve ser incorporada como **covariável**, e não tratada como ruído.

O eixo vertical é deliberadamente livre entre painéis: o que se compara aqui é a **forma** da curva dentro de cada faixa, não o nível, que já é conhecido do Gráfico 1.

### Gráfico 5. Correlação entre variáveis operacionais e a detração

*Tipo:* dois painéis — barras horizontais e matriz de correlação de Spearman, triangular inferior. *Variáveis:* oito numéricas, incluindo o alvo binarizado.

Spearman e não Pearson: todas as numéricas apresentam forte assimetria positiva, como a seção 6.2 mostrou, e o alvo é binário. Separar a correlação com o alvo (painel a) da correlação entre as explicativas (painel b) evita misturar duas perguntas diferentes numa matriz só; a máscara triangular no painel b evita a redundância da matriz simétrica.

In [ ]:
mostrar(g5_correlacao(df), 'g5_correlacao')

cols = ['DETRATOR', 'ESTATISTICA_ATRASOSAIDA', 'ATRASO_CHEGADA', 'TEMPO_VOO',
        'N_TRECHOS', 'QTDE_VIAGENS_12M', 'QTDE_VIAGENS_24M', 'QTDE_VIAGENS_36M']
print('Correlacao de cada variavel com o alvo, em ordem decrescente:')
print(df[cols].corr(method='spearman')['DETRATOR'].drop('DETRATOR')
        .sort_values(ascending=False).round(3).to_string())

**Leitura.** Os painéis confirmam que **nenhuma variável operacional isolada apresenta correlação forte com a detração**. Combinado com os valores de V de Cramér da seção 6.5, o resultado sustenta que a detração é fenômeno multivariado e que a escolha de um classificador não linear se justifica pela ausência de preditor dominante.

O fato de o atraso na chegada superar o atraso na saída é coerente com a experiência do Cliente, já que o custo percebido do atraso se materializa no destino e não no portão de embarque. A observação deve ser lida com a ressalva da seção 7.2: a regra de cálculo de `ATRASO_CHEGADA` ainda aguarda validação do parceiro, e parte da associação pode decorrer da inclusão de tempo de reacomodação em voos cancelados.

Destacam-se dois blocos de colinearidade no painel b. O primeiro, entre os dois campos de atraso. O segundo, mais severo, entre `QTDE_VIAGENS_12M`, `_24M` e `_36M`, o que exigirá selecionar apenas uma das janelas ou construir uma razão entre elas. Há ainda associação entre `TEMPO_VOO` e `N_TRECHOS`, esperada por construção.

### Exportação das figuras

As sete figuras foram gravadas em disco com os mesmos nomes que `documents/documentacao.md` referencia em `assets/`. A célula abaixo confere o que foi escrito.

In [ ]:
if DIR_FIGURAS and Path(DIR_FIGURAS).is_dir():
    for arquivo in sorted(os.listdir(DIR_FIGURAS)):
        caminho = os.path.join(DIR_FIGURAS, arquivo)
        print(f'{arquivo:<40} {os.path.getsize(caminho) / 1024:>7.0f} KB')
    print(f'\nTotal em {os.path.abspath(DIR_FIGURAS)}')
else:
    print('DIR_FIGURAS desativado: as figuras foram apenas exibidas.')

# No Colab, para baixar as sete de uma vez:
# import shutil
# from google.colab import files
# shutil.make_archive('figuras', 'zip', DIR_FIGURAS)
# files.download('figuras.zip')

Basta descompactar o resultado em `assets/`, na raiz do repositório, para que as imagens da seção 4.2.1 passem a resolver.

---

## 10. Síntese e implicações para as próximas fases

*Corresponde ao item (i) da documentação.*

1. **A base é de alta qualidade estrutural.** Junção 1:1 completa e validada em execução entre as três tabelas transacionais, ausência de duplicidades e cobertura temporal de 36 meses.
2. **A amostra não é representativa da população de passageiros.** O viés está correlacionado com o alvo, e a pós-estratificação com a tabela fornecida pela Azul reconcilia o NPS da amostra com a métrica oficial da companhia, com intervalos de confiança que não se sobrepõem aos da amostra bruta.
3. **A operação explica a maior parte da detração, mas não toda.** O piso irredutível em condições operacionais ideais delimita o teto de desempenho realista de um modelo baseado em variáveis de operação.
4. **Duas hipóteses operacionais de alto retorno seguem para validação:** o limiar de 20 a 30 minutos como janela de intervenção preventiva, e a antecedência do aviso de cancelamento como alavanca de mitigação. Ambas são associações observacionais robustas aos controles disponíveis, e nenhuma delas foi estabelecida como efeito causal.

### Restrições que seguem para a modelagem

| Restrição | Origem |
|---|---|
| Partição agrupada por `ID_GOLDENRECORD` | Dependência intracliente, seção 5 |
| Validação com partição temporal | Efeito de período, seção 8.4 |
| Usar apenas uma das três janelas de `QTDE_VIAGENS` | Colinearidade, Gráfico 5 |
| Substituir `SUB_FIL_FREQUENCIAAZUL` por `QTDE_VIAGENS_12M` | Erro de medida, seção 7.3 |
| Excluir `N_ASSENTOS_AUDIT` do conjunto de preditores | Redundância com `N_TRECHOS`, seção 4.2 |
| Excluir os campos `NPS_*` de etapa da jornada | Vazamento de dados, seção 6.1 |
| Tratar `TIER_VIAGEM` e `FAIXA_ATRASO` com interação | Gráfico 3 |
| Reportar incerteza sobre o n efetivo, não sobre o n bruto | Diagnóstico dos pesos, seção 8.3 |

Permanece **uma pendência técnica aberta com o parceiro**: a validação da regra de cálculo de `ATRASO_CHEGADA`.

### Registro completo da execução

O log acumulado por `src/clean.py` documenta cada filtro aplicado, com contagem, e serve de trilha de auditoria da execução.

In [ ]:
print('\n'.join(log))

---

### Referências

CHAPMAN, P. et al. **CRISP-DM 1.0: step-by-step data mining guide**. SPSS Inc., 2000.

CRAMÉR, H. **Mathematical methods of statistics**. Princeton: Princeton University Press, 1946.

GROVES, R. M.; PEYTCHEVA, E. The impact of nonresponse rates on nonresponse bias: a meta-analysis. **Public Opinion Quarterly**, v. 72, n. 2, p. 167-189, 2008.

HUNTER, J. D. Matplotlib: a 2D graphics environment. **Computing in Science & Engineering**, v. 9, n. 3, p. 90-95, 2007.

KISH, L. **Survey sampling**. New York: John Wiley & Sons, 1965.

McKINNEY, W. Data structures for statistical computing in Python. In: **Proceedings of the 9th Python in Science Conference**, p. 56-61, 2010.

REICHHELD, F. F. The one number you need to grow. **Harvard Business Review**, v. 81, n. 12, p. 46-54, 2003.

VALLIANT, R. Post-stratification and conditional variance estimation. **Journal of the American Statistical Association**, v. 88, n. 421, p. 89-96, 1993.

WASKOM, M. L. Seaborn: statistical data visualization. **Journal of Open Source Software**, v. 6, n. 60, 3021, 2021.